# Graded Challenge 3 – Set 2
**Property Market Analysis: Rumah Dijual di Bekasi (DATA DUMMY)**

**Nama:** Nawafi Suren  
**Batch:** HCK-47

Notebook ini mencakup data understanding, cleaning, SMART problem statement, statistik deskriptif, confidence interval, uji hipotesis, korelasi, dan kesimpulan.

> **PERINGATAN:** Semua data adalah sintetis untuk latihan. Tidak ada scraping aktual dari Rumah123. Ini bukan submission yang membuktikan requirement web scraping GC3. Tugas asli tetap meminta minimal 10 halaman website dan minimal 100 listing aktual.

## 1. Import library
Library yang digunakan: pandas/numpy untuk data, matplotlib/seaborn untuk visualisasi, dan SciPy untuk inferensi statistik.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
pd.set_option("display.max_columns", None)

## 2. Simulasi data scraping
CSV berisi 12 halaman simulasi × 10 listing = 120 baris. Dalam tugas asli, tahap ini harus menggunakan request dan BeautifulSoup untuk mengambil halaman Rumah123.

In [ ]:
df_raw = pd.read_csv("P0G3_data_nawafi_suren_DUMMY.csv")
print("Jumlah listing:", len(df_raw))
print("Jumlah halaman simulasi:", df_raw["page"].nunique())
display(df_raw.head())

## 3. Data understanding: beberapa baris
Tampilkan sampel untuk memahami struktur data. Data mentah masih memiliki satuan luas, suffix lokasi, harga berbentuk teks, dan nilai carport kosong.

In [ ]:
display(df_raw.head())
display(df_raw.sample(5, random_state=47))

## 4. Informasi rangkuman data
`info()` memeriksa tipe data dan non-null. `describe(include='all')` memberi ringkasan numerik maupun kategorikal. Selanjutnya teks perlu dibersihkan dan kolom analisis diubah menjadi angka.

In [ ]:
df_raw.info()
display(df_raw.describe(include="all").T)

## 5. Missing value dan duplikasi
Missing carport pada latihan ini sengaja dibuat. Sesuai instruksi, carport kosong dianggap 0. Duplikasi diperiksa menggunakan listing_id.

In [ ]:
display(df_raw.isna().sum().to_frame("missing_count"))
print("Duplikasi listing_id:", df_raw.duplicated(subset=["listing_id"]).sum())

## 6. Data cleaning
Lokasi dibersihkan dari `, Bekasi`; satuan luas dihapus; carport kosong menjadi 0; kamar dan luas dikonversi numerik. Harga promo dipotong pada kata HEMAT dan dikonversi ke juta Rupiah.

In [ ]:
df = df_raw.copy()
df["Lokasi"] = df["Lokasi"].str.replace(r",\s*Bekasi$", "", regex=True).str.strip()
for col in ["Luas Tanah", "Luas Bangunan"]:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(r"[^0-9]", "", regex=True), errors="coerce")
df["Carport"] = pd.to_numeric(df["Carport"], errors="coerce").fillna(0).astype(int)
for col in ["Jumlah Kamar Tidur", "Jumlah Kamar Mandi"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
def harga_ke_juta(value):
    s = str(value).split("HEMAT")[0].strip().replace("Rp","").replace(" ","")
    m = re.search(r"([0-9]+(?:[.,][0-9]+)?)(Miliar|Juta)", s, flags=re.I)
    if not m: return np.nan
    n = float(m.group(1).replace(",", "."))
    return n*1000 if m.group(2).lower()=="miliar" else n
df["Harga Rumah"] = df["Harga Rumah"].apply(harga_ke_juta)
numeric_cols = ["Luas Tanah","Luas Bangunan","Carport","Jumlah Kamar Tidur","Jumlah Kamar Mandi","Harga Rumah"]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
display(df.head())
display(df.dtypes)

## 7. Validasi cleaning
Periksa missing setelah cleaning dan duplikasi. Baris yang tidak memiliki nilai penting untuk analisis dibuang. Outlier tidak otomatis dihapus karena dapat merupakan karakteristik alami rumah.

In [ ]:
display(df.isna().sum().to_frame("missing_after_cleaning"))
df_clean = df.dropna(subset=numeric_cols).copy()
print("Baris bersih:", len(df_clean), "| halaman simulasi:", df_clean["page"].nunique())
display(df_clean.describe().T)

## 8. Business Understanding — SMART

**Problem statement:** Dalam satu sesi analisis sebelum menentukan anggaran pembelian, saya akan mengevaluasi setidaknya 100 listing rumah dari minimal 10 halaman pencarian Rumah123 di Bekasi untuk mengestimasi kisaran rata-rata harga dengan confidence interval 95%, menguji apakah rumah ber-carport memiliki rata-rata harga lebih tinggi daripada rumah tanpa carport, dan mengukur hubungan luas tanah serta luas bangunan dengan harga.

- **Specific:** rumah dijual di Bekasi, fokus harga, carport, luas tanah dan bangunan.
- **Measurable:** minimal 100 listing/10 halaman, CI 95%, p-value dan koefisien korelasi.
- **Achievable:** atribut berasal dari informasi listing.
- **Relevant:** membantu memahami harga dan atribut properti yang berkaitan dengan harga.
- **Time-bound:** diselesaikan dalam satu sesi analisis sebelum menetapkan anggaran.

**Catatan:** pada notebook dummy, halaman dan listing hanya simulasi.

## 9. Statistik deskriptif
Hitung mean, median, standar deviasi, skewness, dan kurtosis untuk semua variabel numerik analisis. Skewness menunjukkan kemencengan distribusi; kurtosis pandas adalah excess kurtosis (normal mendekati 0).

In [ ]:
analysis_cols = ["Luas Tanah","Luas Bangunan","Carport","Jumlah Kamar Tidur","Jumlah Kamar Mandi","Harga Rumah"]
stats_desc = pd.DataFrame({
    "mean":df_clean[analysis_cols].mean(),
    "median":df_clean[analysis_cols].median(),
    "std":df_clean[analysis_cols].std(),
    "skewness":df_clean[analysis_cols].skew(),
    "kurtosis":df_clean[analysis_cols].kurt()
})
display(stats_desc.round(3))

## 10. Distribusi harga
Histogram dan boxplot membantu membaca bentuk distribusi, median, sebaran, serta kandidat outlier. Outlier tidak dihapus hanya karena ekstrem.

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(13,4))
sns.histplot(df_clean["Harga Rumah"], bins=18, kde=False, ax=ax[0])
ax[0].set_title("Distribusi Harga Rumah"); ax[0].set_xlabel("Harga (juta Rupiah)")
sns.boxplot(x=df_clean["Harga Rumah"], ax=ax[1])
ax[1].set_title("Boxplot Harga Rumah"); ax[1].set_xlabel("Harga (juta Rupiah)")
plt.tight_layout(); plt.show()

## 11. Confidence interval 95%

Gunakan t-distribution karena standar deviasi populasi tidak diketahui.

\[
CI = \bar{x} \pm t_{0.975,n-1} \times \frac{s}{\sqrt{n}}
\]

Interval ini mengestimasi rata-rata populasi, bukan rentang harga seluruh rumah individual.

In [ ]:
prices = df_clean["Harga Rumah"].dropna()
n = len(prices); mean_price = prices.mean(); std_price = prices.std(ddof=1)
t_crit = stats.t.ppf(0.975, df=n-1)
margin = t_crit * stats.sem(prices)
ci_lower, ci_upper = mean_price-margin, mean_price+margin
print(f"n={n}; mean={mean_price:.2f} juta")
print(f"95% CI: [{ci_lower:.2f}, {ci_upper:.2f}] juta Rupiah")

## 12. Uji hipotesis: pengaruh keberadaan carport pada harga

Bandingkan rumah dengan carport (`Carport > 0`) dan tanpa carport (`Carport = 0`) menggunakan **Welch independent two-sample t-test**, karena varians kedua kelompok tidak diasumsikan sama.

- H0: μ_carport ≤ μ_tanpa_carport.
- H1: μ_carport > μ_tanpa_carport.
- Jenis: uji beda dua rata-rata independen, satu arah (right-tailed).
- α = 0,05.

SciPy memberikan p-value dua arah. Konversi ke satu arah dilakukan sesuai tanda t-statistic.

In [ ]:
g1 = df_clean.loc[df_clean["Carport"]>0, "Harga Rumah"]
g0 = df_clean.loc[df_clean["Carport"]==0, "Harga Rumah"]
t_stat, p_two = stats.ttest_ind(g1, g0, equal_var=False, nan_policy="omit")
p_one = p_two/2 if t_stat>0 else 1-p_two/2
alpha = 0.05
print("n dengan carport:",len(g1),"| n tanpa:",len(g0))
print("Mean dengan:",round(g1.mean(),2),"juta | mean tanpa:",round(g0.mean(),2),"juta")
print("t-statistic:",round(t_stat,4),"| p-value satu arah:",round(p_one,6))
if p_one < alpha:
    print("Tolak H0: terdapat bukti rata-rata harga kelompok ber-carport lebih tinggi.")
else:
    print("Gagal menolak H0: bukti belum cukup bahwa kelompok ber-carport lebih mahal.")

## 13. Korelasi luas tanah dan luas bangunan dengan harga

Gunakan **Spearman rank correlation** dari SciPy. Spearman cocok untuk menilai hubungan monotonik dan lebih tahan terhadap distribusi menceng/outlier dibanding Pearson. Korelasi bukan bukti sebab-akibat.

- H0: ρ = 0 (tidak ada korelasi monotonik).
- H1: ρ ≠ 0 (ada korelasi monotonik).
- α = 0,05.

In [ ]:
rho_lt, p_lt = stats.spearmanr(df_clean["Luas Tanah"], df_clean["Harga Rumah"])
rho_lb, p_lb = stats.spearmanr(df_clean["Luas Bangunan"], df_clean["Harga Rumah"])
corr = pd.DataFrame({"Variabel":["Luas Tanah","Luas Bangunan"],
                     "Spearman rho":[rho_lt,rho_lb],"p-value":[p_lt,p_lb]})
display(corr.round(4))
if abs(rho_lt)>abs(rho_lb):
    print("Korelasi luas tanah dengan harga lebih kuat pada dataset ini.")
elif abs(rho_lb)>abs(rho_lt):
    print("Korelasi luas bangunan dengan harga lebih kuat pada dataset ini.")
else:
    print("Kekuatan korelasi keduanya sama.")
for name,r,p in [("Luas Tanah",rho_lt,p_lt),("Luas Bangunan",rho_lb,p_lb)]:
    print(f"{name}: rho={r:.3f}, p={p:.4g}, signifikan={p<0.05}")

## 14. Scatterplot pendukung
Scatterplot membantu memeriksa arah dan pola hubungan antara luas dengan harga.

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(13,4))
sns.scatterplot(data=df_clean,x="Luas Tanah",y="Harga Rumah",ax=ax[0])
ax[0].set_title("Luas Tanah vs Harga"); ax[0].set_ylabel("Harga (juta Rupiah)")
sns.scatterplot(data=df_clean,x="Luas Bangunan",y="Harga Rumah",ax=ax[1])
ax[1].set_title("Luas Bangunan vs Harga"); ax[1].set_ylabel("Harga (juta Rupiah)")
plt.tight_layout(); plt.show()

## 15. Kesimpulan berdasarkan output
Gunakan hasil aktual yang tampil saat notebook dijalankan untuk menyusun narasi. Keluaran berikut membantu merangkum angka; seluruh interpretasi hanya berlaku untuk data dummy.

In [ ]:
print(f"Data latihan: {len(df_clean)} listing dari {df_clean['page'].nunique()} halaman simulasi.")
print(f"Rata-rata harga: Rp {mean_price:,.2f} juta.")
print(f"CI 95% rata-rata: Rp {ci_lower:,.2f}–{ci_upper:,.2f} juta.")
print(f"Mean carport={g1.mean():,.2f}; tanpa carport={g0.mean():,.2f} juta.")
print(f"Uji Welch satu arah p={p_one:.4g}.")
print(f"Spearman LT rho={rho_lt:.3f} (p={p_lt:.4g}); LB rho={rho_lb:.3f} (p={p_lb:.4g}).")
print("Semua temuan ini adalah demonstrasi dari data sintetis, bukan gambaran pasar Bekasi.")

## 16. Checklist sebelum submission resmi

- [ ] Implementasikan scraping aktual Rumah123 menggunakan BeautifulSoup.
- [ ] Buktikan minimal 10 halaman website benar-benar diakses.
- [ ] Kumpulkan minimal 100 listing aktual.
- [ ] Verifikasi selector HTML terbaru karena website dinamis.
- [ ] Simpan data aktual sebagai `P0G3_data_nawafi_suren.csv`.
- [ ] Jalankan seluruh notebook dari awal hingga akhir tanpa error di VS Code.
- [ ] Perbarui insight, angka, dan kesimpulan berdasarkan data aktual.
- [ ] Jangan mengklaim dataset dummy sebagai hasil scraping.